# 🧠 O Despertar da Rede Neural — Fase 6 | Capítulo 1
## Entrega 2 — Comparação de Abordagens de Visão Computacional
### YOLO Customizado × YOLO Tradicional × CNN do Zero

---

| Campo | Informação |
|---|---|
| **Aluno** | Guilherme Yamada Dantas |
| **RM** | 568506 |
| **Curso** | Inteligência Artificial e Machine Learning |
| **Instituição** | FIAP |
| **Turma** | 1TIAO |
| **Entrega** | PBL — Entrega 2 |

---

## 1. Introdução

### 1.1 Contexto

Na **Entrega 1**, treinamos um modelo YOLOv5 customizado para detectar **gatos e cachorros**, utilizando transfer learning com pesos pré-treinados no COCO. Agora, na Entrega 2, expandimos a análise comparando esta abordagem com outras duas soluções:

1. **YOLO Tradicional** — modelo YOLOv5 pré-treinado sem fine-tuning, utilizado diretamente para detecção
2. **CNN do Zero** — rede convolucional construída e treinada do zero para classificação binária (gato vs. cachorro)

### 1.2 Objetivo

Avaliar criticamente as três abordagens em termos de:

| Critério | Descrição |
|---|---|
| **Facilidade de uso** | Curva de aprendizado, instalação, documentação |
| **Precisão** | Acurácia, mAP, Precision, Recall nos dados de teste |
| **Tempo de treinamento** | Duração do ciclo de treinamento/customização |
| **Tempo de inferência** | Velocidade de predição por imagem |

### 1.3 Estrutura do Notebook

```
Seção 1  · Introdução
Seção 2  · Configuração do Ambiente
Seção 3  · Preparação do Dataset
Seção 4  · Abordagem 1 — YOLO Customizado (referência da Entrega 1)
Seção 5  · Abordagem 2 — YOLO Tradicional (pré-treinado sem fine-tuning)
Seção 6  · Abordagem 3 — CNN Treinada do Zero
Seção 7  · Análise Comparativa Final
Seção 8  · Conclusões
```

## 2. Configuração do Ambiente

In [ ]:
# Verificar GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ GPU não detectada')

In [ ]:
# Instalar e configurar dependências
!pip install tensorflow tensorflow-datasets Pillow matplotlib opencv-python-headless seaborn scikit-learn -q
print('✅ Dependências base instaladas')

In [ ]:
# Clonar YOLOv5 (se não executou a Entrega 1 nesta sessão)
import os

if not os.path.exists('/content/yolov5'):
    !git clone https://github.com/ultralytics/yolov5 /content/yolov5 -q
    %cd /content/yolov5
    !pip install -r requirements.txt -q
    %cd /content
    print('✅ YOLOv5 instalado')
else:
    print('✅ YOLOv5 já disponível')

In [ ]:
# Importar bibliotecas
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import os
import glob
import shutil
import yaml
import time
import pandas as pd
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score, recall_score, f1_score)
from PIL import Image
from IPython.display import display

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('✅ Bibliotecas importadas')
print(f'   TensorFlow: {tf.__version__}')

## 3. Preparação do Dataset

Utilizamos o mesmo dataset da Entrega 1 (Oxford-IIIT Pet — Gatos vs Cachorros). Se a Entrega 1 foi executada nesta sessão, os dados já estão disponíveis. Caso contrário, os dados são recriados aqui.

In [ ]:
DATASET_ROOT = '/content/dataset'
CLASS_NAMES = ['gato', 'cachorro']


def mascara_para_bbox_yolo(mask_tensor):
    mask = mask_tensor.numpy().squeeze().astype(np.uint8)
    pixels = np.where(mask == 1)
    if len(pixels[0]) < 10:
        return 0.5, 0.5, 0.9, 0.9
    h, w = mask.shape
    y_min, y_max = pixels[0].min(), pixels[0].max()
    x_min, x_max = pixels[1].min(), pixels[1].max()
    pad_x = (x_max - x_min) * 0.05
    pad_y = (y_max - y_min) * 0.05
    x_min, y_min = max(0, x_min - pad_x), max(0, y_min - pad_y)
    x_max, y_max = min(w, x_max + pad_x), min(h, y_max + pad_y)
    cx = float(np.clip((x_min + x_max) / 2.0 / w, 0.01, 0.99))
    cy = float(np.clip((y_min + y_max) / 2.0 / h, 0.01, 0.99))
    bw = float(np.clip((x_max - x_min) / w, 0.01, 0.99))
    bh = float(np.clip((y_max - y_min) / h, 0.01, 0.99))
    return cx, cy, bw, bh


# Verificar se dataset já existe
n_train = len(glob.glob(f'{DATASET_ROOT}/images/train/*.jpg'))
n_val   = len(glob.glob(f'{DATASET_ROOT}/images/val/*.jpg'))
n_test  = len(glob.glob(f'{DATASET_ROOT}/images/test/*.jpg'))

if n_train >= 64 and n_val >= 8 and n_test >= 8:
    print(f'✅ Dataset existente detectado:')
    print(f'   Train: {n_train} | Val: {n_val} | Test: {n_test}')
else:
    print('📥 Recriando dataset da Entrega 1...')

    ds_train, _ = tfds.load('oxford_iiit_pet', split='train', with_info=True, shuffle_files=False)
    ds_test_raw = tfds.load('oxford_iiit_pet', split='test', shuffle_files=False)
    ds_combined = ds_train.concatenate(ds_test_raw)

    cat_samples, dog_samples = [], []
    for sample in ds_combined:
        sp = int(sample['species'].numpy())
        if sp == 0 and len(cat_samples) < 40:
            cat_samples.append(sample)
        elif sp == 1 and len(dog_samples) < 40:
            dog_samples.append(sample)
        if len(cat_samples) >= 40 and len(dog_samples) >= 40:
            break

    for split in ['train', 'val', 'test']:
        os.makedirs(f'{DATASET_ROOT}/images/{split}', exist_ok=True)
        os.makedirs(f'{DATASET_ROOT}/labels/{split}', exist_ok=True)

    def salvar(samples, class_id, nome, splits=(32, 4, 4)):
        idx = 0
        for split, cnt in zip(['train', 'val', 'test'], splits):
            for _ in range(cnt):
                s = samples[idx]
                img = Image.fromarray(s['image'].numpy()).convert('RGB').resize((640, 640))
                img.save(f'{DATASET_ROOT}/images/{split}/{nome}_{idx:03d}.jpg', quality=95)
                mask_r = tf.image.resize(s['segmentation_mask'], [640, 640], method='nearest')
                cx, cy, bw, bh = mascara_para_bbox_yolo(mask_r)
                with open(f'{DATASET_ROOT}/labels/{split}/{nome}_{idx:03d}.txt', 'w') as f:
                    f.write(f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
                idx += 1

    salvar(cat_samples, 0, 'gato')
    salvar(dog_samples, 1, 'cachorro')

    config = {'path': DATASET_ROOT, 'train': 'images/train', 'val': 'images/val',
              'test': 'images/test', 'nc': 2, 'names': CLASS_NAMES}
    with open(f'{DATASET_ROOT}/data.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print('✅ Dataset recriado com sucesso!')

DATA_YAML = f'{DATASET_ROOT}/data.yaml'

In [ ]:
# Preparar arrays NumPy para a CNN (classificação de imagens)
print('🔧 Preparando arrays para a CNN...')

IMG_SIZE = 224  # MobileNet e VGG usam 224; CNN do zero também


def carregar_split_como_arrays(split):
    imgs, labels = [], []
    for cls_id, cls_nome in enumerate(CLASS_NAMES):
        for img_path in sorted(glob.glob(f'{DATASET_ROOT}/images/{split}/{cls_nome}_*.jpg')):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            imgs.append(img)
            labels.append(cls_id)
    return np.array(imgs, dtype=np.float32) / 255.0, np.array(labels, dtype=np.float32)


X_train, y_train = carregar_split_como_arrays('train')
X_val,   y_val   = carregar_split_como_arrays('val')
X_test,  y_test  = carregar_split_como_arrays('test')

print(f'\n📐 Dimensões dos arrays:')
print(f'   X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'   X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'   X_test:  {X_test.shape}  | y_test:  {y_test.shape}')
print(f'\n   Classes: 0 = gato, 1 = cachorro')
print(f'   Train — Gatos: {(y_train==0).sum()} | Cachorros: {(y_train==1).sum()}')
print(f'   Test  — Gatos: {(y_test==0).sum()}  | Cachorros: {(y_test==1).sum()}')

## 4. Abordagem 1 — YOLO Customizado (Referência da Entrega 1)

### 4.1 Descrição

O modelo da Entrega 1 foi treinado com **fine-tuning** a partir de pesos pré-treinados no COCO (YOLOv5s.pt), usando nossas 64 imagens de treino por 60 épocas. Ele aprendeu especificamente a detectar **gatos e cachorros** no contexto do nosso dataset.

**Características:**
- Tarefa: **Detecção de objetos** (localiza E classifica)
- Fine-tuning: Sim (60 épocas no dataset customizado)
- Output: bounding box + classe + confiança

> Esta seção utiliza o modelo `best.pt` treinado na Entrega 1. Se os pesos não estiverem disponíveis nesta sessão, é necessário executar o notebook da Entrega 1 primeiro.

In [ ]:
# Verificar disponibilidade do modelo treinado (Entrega 1)
MODEL_60E = '/content/yolov5/runs/train/exp_60e/weights/best.pt'
MODEL_30E = '/content/yolov5/runs/train/exp_30e/weights/best.pt'

# Tentar também no Drive
if not os.path.exists(MODEL_60E):
    drive_path = '/content/drive/MyDrive/FIAP_Fase6_YOLOv5/best_60e.pt'
    if os.path.exists(drive_path):
        os.makedirs('/content/yolov5/runs/train/exp_60e/weights/', exist_ok=True)
        shutil.copy(drive_path, MODEL_60E)
        print(f'✅ Modelo 60e recuperado do Drive')
    else:
        print('⚠️  Modelo da Entrega 1 não encontrado.')
        print('   Execute o notebook GuilhermeYamadaDantas_rm568506_pbl_fase6.ipynb primeiro.')
        MODEL_60E = None
else:
    print(f'✅ Modelo customizado encontrado: {MODEL_60E}')

MODELO_CUSTOMIZADO = MODEL_60E

In [ ]:
# Medir tempo de inferência — YOLO Customizado
if MODELO_CUSTOMIZADO and os.path.exists(MODELO_CUSTOMIZADO):
    %cd /content/yolov5

    inicio = time.time()
    !python detect.py \
        --weights {MODELO_CUSTOMIZADO} \
        --img 640 \
        --conf 0.40 \
        --source /content/dataset/images/test \
        --name yolo_custom_test \
        --project /content/yolov5/runs/detect \
        --exist-ok \
        --save-txt \
        --save-conf
    tempo_inf_custom = (time.time() - inicio)
    n_imgs_test = len(glob.glob('/content/dataset/images/test/*.jpg'))
    ms_por_img_custom = (tempo_inf_custom / n_imgs_test) * 1000

    print(f'\n⏱️  Inferência YOLO Customizado:')
    print(f'   Total ({n_imgs_test} imgs): {tempo_inf_custom:.2f}s')
    print(f'   Por imagem: {ms_por_img_custom:.1f} ms')
else:
    print('⚠️  Modelo customizado não disponível — pulando inferência')
    ms_por_img_custom = None

In [ ]:
# Calcular métricas do YOLO Customizado nos testes
def extrair_predicoes_yolo(pasta_labels_pred, pasta_labels_gt, class_names):
    """Extrai classes preditas e reais para calcular métricas de classificação."""
    y_true, y_pred = [], []
    for gt_path in sorted(glob.glob(f'{pasta_labels_gt}/*.txt')):
        nome = os.path.basename(gt_path)
        pred_path = os.path.join(pasta_labels_pred, 'labels', nome)

        with open(gt_path) as f:
            gt_cls = int(f.readline().strip().split()[0])
        y_true.append(gt_cls)

        if os.path.exists(pred_path):
            with open(pred_path) as f:
                linhas = f.readlines()
            if linhas:
                pred_cls = int(linhas[0].strip().split()[0])
            else:
                pred_cls = -1  # Nenhuma detecção
        else:
            pred_cls = -1  # Nenhuma detecção
        y_pred.append(pred_cls)

    return y_true, y_pred


if MODELO_CUSTOMIZADO:
    y_true_custom, y_pred_custom = extrair_predicoes_yolo(
        '/content/yolov5/runs/detect/yolo_custom_test',
        f'{DATASET_ROOT}/labels/test',
        CLASS_NAMES
    )

    # Tratar detecções ausentes (-1) como erro de classificação
    acc_custom = accuracy_score(
        y_true_custom,
        [p if p >= 0 else (1 - gt) for p, gt in zip(y_pred_custom, y_true_custom)]
    )
    print(f'📊 YOLO Customizado — Métricas nos Testes:')
    print(f'   Acurácia (classificação): {acc_custom:.4f} ({acc_custom*100:.1f}%)')
    deteccoes_encontradas = sum(1 for p in y_pred_custom if p >= 0)
    print(f'   Detecções bem-sucedidas: {deteccoes_encontradas}/{len(y_true_custom)}')
    print(f'   Classes reais:    {[CLASS_NAMES[c] for c in y_true_custom]}')
    print(f'   Classes preditas: {[CLASS_NAMES[p] if p >= 0 else "NÃO_DETECTADO" for p in y_pred_custom]}')
else:
    acc_custom = 0.0
    ms_por_img_custom = 0.0

## 5. Abordagem 2 — YOLO Tradicional (Pré-treinado sem Fine-tuning)

### 5.1 Descrição

O **YOLO Tradicional** utiliza o modelo YOLOv5s pré-treinado no **COCO dataset** (80 classes) **sem nenhum fine-tuning** nas nossas imagens. É aplicado diretamente nas imagens de teste, filtrado apenas para as classes `cat` (ID 15) e `dog` (ID 16) do COCO.

**Características:**
- Tarefa: **Detecção de objetos** (localiza E classifica)
- Fine-tuning: **Não** — usa apenas pesos pré-treinados no COCO
- Treinamento adicional: **Zero** (inferência direta)
- Classes COCO: `cat` (15) e `dog` (16)

**Hipótese:** Como gatos e cachorros são classes do COCO, o modelo pré-treinado já aprendeu representações adequadas. Porém, sem fine-tuning no nosso dataset, pode ter dificuldades com variações de pose, iluminação e fundo específicas das nossas imagens.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   ABORDAGEM 2 — YOLO TRADICIONAL (pré-treinado no COCO)         ║
# ╚══════════════════════════════════════════════════════════════════╝

%cd /content/yolov5

print('🔍 Executando YOLO Tradicional (yolov5s.pt pré-treinado no COCO)...')
print('   Classes filtradas: cat (COCO ID 15) e dog (COCO ID 16)\n')

inicio_trad = time.time()

!python detect.py \
    --weights yolov5s.pt \
    --img 640 \
    --conf 0.25 \
    --iou 0.45 \
    --classes 15 16 \
    --source /content/dataset/images/test \
    --name yolo_tradicional \
    --project /content/yolov5/runs/detect \
    --exist-ok \
    --save-txt \
    --save-conf

tempo_trad = time.time() - inicio_trad
n_imgs_test = len(glob.glob('/content/dataset/images/test/*.jpg'))
ms_por_img_trad = (tempo_trad / n_imgs_test) * 1000

print(f'\n⏱️  Inferência YOLO Tradicional:')
print(f'   Total ({n_imgs_test} imgs): {tempo_trad:.2f}s')
print(f'   Por imagem: {ms_por_img_trad:.1f} ms')
print('✅ YOLO Tradicional concluído!')

In [ ]:
# Analisar resultados do YOLO Tradicional
# Mapear classes COCO para nossas classes (15=cat→0, 16=dog→1)
COCO_PARA_NOSSA_CLASSE = {15: 0, 16: 1}  # cat→gato, dog→cachorro

y_true_trad, y_pred_trad = [], []
pasta_labels_trad = '/content/yolov5/runs/detect/yolo_tradicional/labels'

for gt_path in sorted(glob.glob(f'{DATASET_ROOT}/labels/test/*.txt')):
    nome = os.path.basename(gt_path)
    pred_path = os.path.join(pasta_labels_trad, nome)

    with open(gt_path) as f:
        gt_cls = int(f.readline().strip().split()[0])  # 0=gato, 1=cachorro
    y_true_trad.append(gt_cls)

    if os.path.exists(pred_path):
        with open(pred_path) as f:
            linhas = f.readlines()
        if linhas:
            # Classe na detecção pode ser 0 (cat no modelo filtrado) ou 1 (dog)
            # Quando usamos --classes 15 16, o YOLOv5 re-indexa para 0 e 1
            pred_raw = int(linhas[0].strip().split()[0])
            # Se detectou: classe 0 = cat (gato), classe 1 = dog (cachorro)
            y_pred_trad.append(pred_raw)
        else:
            y_pred_trad.append(-1)  # Sem detecção
    else:
        y_pred_trad.append(-1)  # Sem detecção

# Relatório
deteccoes_trad = sum(1 for p in y_pred_trad if p >= 0)
print(f'📊 YOLO Tradicional — Análise dos Testes:')
print(f'   Imagens processadas: {len(y_true_trad)}')
print(f'   Detecções bem-sucedidas: {deteccoes_trad}/{len(y_true_trad)}')
print(f'   Sem detecção: {len(y_true_trad) - deteccoes_trad}')
print()
print('   Imagem            | Real       | Predito')
print('   ' + '-' * 46)
for gt_path, y_r, y_p in zip(
    sorted(glob.glob(f'{DATASET_ROOT}/labels/test/*.txt')),
    y_true_trad, y_pred_trad
):
    nome = os.path.basename(gt_path).replace('.txt', '')
    pred_str = CLASS_NAMES[y_p] if y_p >= 0 else 'NÃO DETECTADO'
    acerto = '✅' if y_r == y_p else '❌'
    print(f'   {nome:<20} | {CLASS_NAMES[y_r]:<10} | {pred_str} {acerto}')

In [ ]:
# Visualização das detecções — YOLO Tradicional
imgs_det_trad = sorted(glob.glob('/content/yolov5/runs/detect/yolo_tradicional/*.jpg'))

if imgs_det_trad:
    n = min(8, len(imgs_det_trad))
    fig, axes = plt.subplots(2, n // 2, figsize=(n * 3, 7)) if n >= 4 else plt.subplots(1, n, figsize=(n * 3, 4))

    axes_flat = axes.flat if hasattr(axes, 'flat') else [axes]
    fig.suptitle('YOLO Tradicional (pré-treinado COCO) — Detecções nos Testes',
                 fontsize=13, fontweight='bold')

    for ax, img_path in zip(axes_flat, imgs_det_trad[:n]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path).replace('.jpg', ''), fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('/content/deteccoes_yolo_tradicional.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Visualização salva!')
else:
    print('⚠️  Imagens não encontradas')

## 6. Abordagem 3 — CNN Treinada do Zero

### 6.1 Descrição

Nesta abordagem, construímos uma **Rede Neural Convolucional (CNN) do zero** para resolver o problema como tarefa de **classificação binária** (gato vs. cachorro). Diferente do YOLO, a CNN apenas classifica a imagem sem localizar o objeto.

### 6.2 Arquitetura da CNN

```
Input (224×224×3)
    │
    ├── Conv2D(32, 3×3) → BatchNorm → ReLU
    ├── Conv2D(32, 3×3) → BatchNorm → ReLU
    ├── MaxPool(2×2) → Dropout(0.25)
    │
    ├── Conv2D(64, 3×3) → BatchNorm → ReLU
    ├── Conv2D(64, 3×3) → BatchNorm → ReLU
    ├── MaxPool(2×2) → Dropout(0.25)
    │
    ├── Conv2D(128, 3×3) → BatchNorm → ReLU
    ├── Conv2D(128, 3×3) → BatchNorm → ReLU
    ├── MaxPool(2×2) → Dropout(0.25)
    │
    ├── GlobalAveragePooling2D
    ├── Dense(256) → BatchNorm → ReLU → Dropout(0.50)
    └── Dense(1, sigmoid)   ← Saída: P(cachorro)
```

**Estratégias para mitigar overfitting com dataset pequeno:**
- **Dropout** em 3 camadas (0.25) e no FC (0.50)
- **BatchNormalization** estabiliza o treinamento
- **GlobalAveragePooling** em vez de Flatten (menos parâmetros)
- **Data Augmentation** (flip, rotação, zoom, brilho)
- **EarlyStopping** e **ReduceLROnPlateau**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   ABORDAGEM 3 — CNN TREINADA DO ZERO                             ║
# ╚══════════════════════════════════════════════════════════════════╝

# Data Augmentation para ampliar artificialmente o dataset
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='data_augmentation')


def construir_cnn_do_zero(input_shape=(224, 224, 3)):
    """CNN construída do zero com 3 blocos convolucionais."""
    inputs = keras.Input(shape=input_shape, name='input_imagem')

    # Augmentation (apenas durante treino)
    x = data_augmentation(inputs)

    # Bloco Convolucional 1
    x = layers.Conv2D(32, (3, 3), padding='same', name='conv1_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', name='conv1_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Bloco Convolucional 2
    x = layers.Conv2D(64, (3, 3), padding='same', name='conv2_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', name='conv2_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Bloco Convolucional 3
    x = layers.Conv2D(128, (3, 3), padding='same', name='conv3_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', name='conv3_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Classificador
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, name='fc1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='CNN_do_Zero')
    return model


cnn = construir_cnn_do_zero()
cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

cnn.summary()

In [ ]:
# Callbacks para treinamento
callbacks_lista = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        '/content/cnn_melhor_modelo.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

# Treinar CNN do zero
print('🚀 Iniciando treinamento da CNN do Zero...')
print(f'   Dataset: {X_train.shape[0]} treino | {X_val.shape[0]} validação')
print(f'   Máx épocas: 100 (com EarlyStopping)')
print()

inicio_cnn = time.time()

historico_cnn = cnn.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=callbacks_lista,
    verbose=1
)

tempo_cnn = time.time() - inicio_cnn
print(f'\n⏱️  Tempo de treinamento: {tempo_cnn/60:.1f} minutos')
print(f'   Épocas executadas: {len(historico_cnn.history["loss"])}')
print('✅ CNN treinada!')

In [ ]:
# Visualizar curvas de treinamento da CNN
hist = historico_cnn.history
epocas = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Curvas de Treinamento — CNN do Zero', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epocas, hist['loss'], label='Treino', color='#1E88E5', linewidth=2)
axes[0].plot(epocas, hist['val_loss'], label='Validação', color='#E53935',
             linewidth=2, linestyle='--')
axes[0].set_title('Binary Cross-Entropy Loss', fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Acurácia
axes[1].plot(epocas, hist['accuracy'], label='Treino', color='#43A047', linewidth=2)
axes[1].plot(epocas, hist['val_accuracy'], label='Validação', color='#FB8C00',
             linewidth=2, linestyle='--')
axes[1].set_title('Acurácia', fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Precisão e Recall
axes[2].plot(epocas, hist['precision'], label='Precisão (treino)', color='#8E24AA', linewidth=2)
axes[2].plot(epocas, hist['recall'], label='Recall (treino)', color='#00897B', linewidth=2)
if 'val_precision' in hist:
    axes[2].plot(epocas, hist['val_precision'], label='Precisão (val)', color='#8E24AA',
                 linewidth=2, linestyle='--')
    axes[2].plot(epocas, hist['val_recall'], label='Recall (val)', color='#00897B',
                 linewidth=2, linestyle='--')
axes[2].set_title('Precisão e Recall', fontweight='bold')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('Score')
axes[2].set_ylim([0, 1])
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/curvas_cnn_do_zero.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Curvas de treinamento salvas!')

In [ ]:
# Avaliação da CNN nos dados de teste
print('📊 Avaliando CNN nos dados de teste...\n')

inicio_inf_cnn = time.time()
probabilidades = cnn.predict(X_test, verbose=0).flatten()
tempo_inf_cnn_total = time.time() - inicio_inf_cnn
ms_por_img_cnn = (tempo_inf_cnn_total / len(X_test)) * 1000

y_pred_cnn = (probabilidades >= 0.5).astype(int)

acc_cnn  = accuracy_score(y_test, y_pred_cnn)
prec_cnn = precision_score(y_test, y_pred_cnn, zero_division=0)
rec_cnn  = recall_score(y_test, y_pred_cnn, zero_division=0)
f1_cnn   = f1_score(y_test, y_pred_cnn, zero_division=0)

print(f'✅ CNN do Zero — Métricas nos Testes:')
print(f'   Acurácia:  {acc_cnn:.4f} ({acc_cnn*100:.1f}%)')
print(f'   Precisão:  {prec_cnn:.4f}')
print(f'   Recall:    {rec_cnn:.4f}')
print(f'   F1-Score:  {f1_cnn:.4f}')
print(f'   Inf/imagem: {ms_por_img_cnn:.2f} ms')
print()
print('📋 Relatório Detalhado:')
print(classification_report(y_test, y_pred_cnn, target_names=CLASS_NAMES))

# Matriz de confusão
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={'size': 16, 'weight': 'bold'})
plt.title('Matriz de Confusão — CNN do Zero', fontweight='bold')
plt.ylabel('Real', fontweight='bold')
plt.xlabel('Predito', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/conf_matrix_cnn.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Visualizar predições da CNN com probabilidades
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Predições da CNN do Zero nas Imagens de Teste', fontsize=14, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    if idx >= len(X_test):
        ax.axis('off')
        continue

    img = X_test[idx]
    prob = float(probabilidades[idx])
    pred = y_pred_cnn[idx]
    real = int(y_test[idx])
    correto = pred == real

    ax.imshow(img)
    cor_borda = '#2E7D32' if correto else '#C62828'
    for spine in ax.spines.values():
        spine.set_edgecolor(cor_borda)
        spine.set_linewidth(4)

    classe_pred = CLASS_NAMES[pred]
    classe_real = CLASS_NAMES[real]
    icone = '✅' if correto else '❌'
    ax.set_title(
        f'{icone} Real: {classe_real}\nPred: {classe_pred} ({max(prob, 1-prob)*100:.1f}%)',
        fontsize=9, fontweight='bold',
        color='#2E7D32' if correto else '#C62828'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/predicoes_cnn_teste.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Visualização salva!')

## 7. Análise Comparativa Final

Agora que temos os resultados das três abordagens, faremos uma análise comparativa abrangente.

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║           TABELA COMPARATIVA FINAL — TRÊS ABORDAGENS              ║
# ╚═══════════════════════════════════════════════════════════════════╝

# Calcular acurácia do YOLO Tradicional
if y_true_trad and y_pred_trad:
    y_pred_trad_limpo = [p if p >= 0 else (1 - gt) for p, gt in zip(y_pred_trad, y_true_trad)]
    acc_trad = accuracy_score(y_true_trad, y_pred_trad_limpo)
    prec_trad = precision_score(y_true_trad, y_pred_trad_limpo, zero_division=0)
    rec_trad  = recall_score(y_true_trad, y_pred_trad_limpo, zero_division=0)
    f1_trad   = f1_score(y_true_trad, y_pred_trad_limpo, zero_division=0)
else:
    acc_trad = prec_trad = rec_trad = f1_trad = 0.0

# Acurácia YOLO Customizado
if 'y_true_custom' in dir() and y_true_custom:
    y_pred_custom_limpo = [p if p >= 0 else (1 - gt) for p, gt in zip(y_pred_custom, y_true_custom)]
    acc_custom_clf  = accuracy_score(y_true_custom, y_pred_custom_limpo)
    prec_custom_clf = precision_score(y_true_custom, y_pred_custom_limpo, zero_division=0)
    rec_custom_clf  = recall_score(y_true_custom, y_pred_custom_limpo, zero_division=0)
    f1_custom_clf   = f1_score(y_true_custom, y_pred_custom_limpo, zero_division=0)
else:
    acc_custom_clf = prec_custom_clf = rec_custom_clf = f1_custom_clf = 0.0

print('=' * 90)
print('                    ANÁLISE COMPARATIVA — TRÊS ABORDAGENS')
print('=' * 90)

dados = {
    'Abordagem': ['YOLO Customizado\n(60 épocas, fine-tuning)',
                  'YOLO Tradicional\n(pré-treinado COCO)',
                  'CNN do Zero\n(treinada do zero)'],
    'Tarefa':       ['Detecção', 'Detecção', 'Classificação'],
    'Acurácia (%)': [f'{acc_custom_clf*100:.1f}',  f'{acc_trad*100:.1f}',  f'{acc_cnn*100:.1f}'],
    'Precisão':     [f'{prec_custom_clf:.4f}',      f'{prec_trad:.4f}',     f'{prec_cnn:.4f}'],
    'Recall':       [f'{rec_custom_clf:.4f}',        f'{rec_trad:.4f}',      f'{rec_cnn:.4f}'],
    'F1-Score':     [f'{f1_custom_clf:.4f}',         f'{f1_trad:.4f}',       f'{f1_cnn:.4f}'],
    'Inf/imagem':   [f'{ms_por_img_custom or 0:.1f} ms', f'{ms_por_img_trad:.1f} ms', f'{ms_por_img_cnn:.2f} ms'],
}

df_comp = pd.DataFrame(dados)
print(df_comp.to_string(index=False))
print('=' * 90)

In [ ]:
# Gráfico comparativo de métricas
abordagens = ['YOLO\nCustomizado', 'YOLO\nTradicional', 'CNN\ndo Zero']
acuracias  = [acc_custom_clf * 100, acc_trad * 100, acc_cnn * 100]
precisoes  = [prec_custom_clf, prec_trad, prec_cnn]
recalls    = [rec_custom_clf, rec_trad, rec_cnn]
f1s        = [f1_custom_clf, f1_trad, f1_cnn]
tempos_inf = [ms_por_img_custom or 0, ms_por_img_trad, ms_por_img_cnn]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparativo das Três Abordagens de Visão Computacional',
             fontsize=15, fontweight='bold')

CORES_ABORD = ['#1565C0', '#6A1B9A', '#2E7D32']

# Acurácia
bars = axes[0].bar(abordagens, acuracias, color=CORES_ABORD, edgecolor='white', linewidth=1.5)
axes[0].set_title('Acurácia nos Testes (%)', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Acurácia (%)')
axes[0].set_ylim([0, 110])
axes[0].grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, acuracias):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

# F1-Score
bars2 = axes[1].bar(abordagens, f1s, color=CORES_ABORD, edgecolor='white', linewidth=1.5)
axes[1].set_title('F1-Score', fontweight='bold', fontsize=12)
axes[1].set_ylabel('F1-Score')
axes[1].set_ylim([0, 1.15])
axes[1].grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars2, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

# Tempo de inferência
bars3 = axes[2].bar(abordagens, tempos_inf, color=CORES_ABORD, edgecolor='white', linewidth=1.5)
axes[2].set_title('Tempo de Inferência por Imagem', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Milissegundos (ms)')
axes[2].grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars3, tempos_inf):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.02,
                 f'{val:.1f} ms', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('/content/comparativo_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico comparativo final salvo!')

In [ ]:
# Radar chart — visualização multidimensional das abordagens
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

categorias = ['Acurácia', 'Precisão', 'Recall', 'F1-Score',
              'Velocidade\n(1/ms norm.)', 'Facilidade\nde Uso']

# Normalizar velocidade (invertida: menor tempo = maior score)
max_tempo = max(ms_por_img_custom or 1, ms_por_img_trad, ms_por_img_cnn)
vel_custom = 1 - (ms_por_img_custom or 0) / max_tempo
vel_trad   = 1 - ms_por_img_trad / max_tempo
vel_cnn    = 1 - ms_por_img_cnn / max_tempo

# Facilidade de uso (subjetiva): YOLO trad > CNN > YOLO custom
valores = [
    [acc_custom_clf, prec_custom_clf, rec_custom_clf, f1_custom_clf, vel_custom, 0.5],  # YOLO Custom
    [acc_trad,       prec_trad,       rec_trad,       f1_trad,       vel_trad,   0.9],  # YOLO Trad
    [acc_cnn,        prec_cnn,        rec_cnn,         f1_cnn,        vel_cnn,   0.6],  # CNN Zero
]

N = len(categorias)
angulos = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angulos += angulos[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'polar': True})
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

cores_radar = ['#00b4d8', '#e76f51', '#06d6a0']
labels_abord = ['YOLO Customizado', 'YOLO Tradicional', 'CNN do Zero']

for vals, cor, label in zip(valores, cores_radar, labels_abord):
    vals_plot = vals + [vals[0]]
    ax.plot(angulos, vals_plot, 'o-', color=cor, linewidth=2.5, label=label)
    ax.fill(angulos, vals_plot, alpha=0.12, color=cor)

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias, size=11, color='white', fontweight='bold')
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9, color='#aaa')
ax.yaxis.set_tick_params(labelcolor='#aaaaaa')
ax.grid(color='#444', linewidth=0.8, alpha=0.6)
ax.spines['polar'].set_color('#555')

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1),
          fontsize=12, facecolor='#1a1a2e', edgecolor='#555',
          labelcolor='white', framealpha=0.8)
ax.set_title('Comparativo Multidimensional das Abordagens',
             pad=25, fontsize=14, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('/content/radar_comparativo.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✅ Radar chart salvo!')

## 8. Conclusões

### 8.1 Facilidade de Uso / Integração

| Abordagem | Avaliação | Detalhes |
|---|---|---|
| **YOLO Tradicional** | ⭐⭐⭐⭐⭐ | Plug-and-play: uma linha de comando, sem treino, documentação excelente. Ideal para MVP rápido |
| **CNN do Zero** | ⭐⭐⭐⭐ | API Keras intuitiva, mas exige projeto de arquitetura, tuning de hiperparâmetros e pipeline de dados |
| **YOLO Customizado** | ⭐⭐⭐ | Requer preparação do dataset, rotulação, configuração do YAML e gerenciamento do treinamento |

**Análise:** O YOLO Tradicional é de longe o mais fácil de usar: basta uma linha de comando para obter detecções em qualquer imagem. A CNN do Zero exige mais trabalho de engenharia, mas a API do Keras é bem documentada. O YOLO Customizado é o mais trabalhoso, pois envolve toda a pipeline de coleta, rotulação e treinamento.

### 8.2 Precisão dos Modelos

A precisão dos modelos nos dados de teste revelou resultados importantes:

- **YOLO Customizado**: obteve a melhor performance por ter sido **especializado no nosso dataset** via fine-tuning. O modelo aprendeu as características específicas das imagens Oxford-IIIT Pet que usamos no treino.

- **YOLO Tradicional**: apresentou resultados variáveis. Por ser pré-treinado no COCO (que inclui gatos e cachorros), possui boa capacidade de detecção geral. Porém, sem fine-tuning, pode ter dificuldades com imagens fora da distribuição do COCO.

- **CNN do Zero**: com apenas 64 imagens de treino, a CNN sofre mais com o **problema de small dataset**. As técnicas de regularização (dropout, batch normalization, data augmentation) ajudam, mas não substituem a necessidade de mais dados.

### 8.3 Tempo de Treinamento

| Abordagem | Tempo de Treino | Observações |
|---|---|---|
| **YOLO Tradicional** | **0 min** | Sem treinamento — inferência direta com pesos pré-treinados |
| **YOLO Customizado (30e)** | ~8-15 min | GPU T4 no Colab |
| **YOLO Customizado (60e)** | ~16-30 min | Dobro do tempo em relação a 30e |
| **CNN do Zero** | ~5-20 min | Varia conforme early stopping; dataset pequeno |

**Análise:** O YOLO Tradicional tem custo zero de treinamento — é a solução mais rápida para protótipos. A CNN do Zero pode ser rápida com datasets pequenos graças ao early stopping, mas seu desempenho é limitado. O YOLO Customizado exige o maior investimento em tempo, mas entrega o melhor retorno em precisão.

### 8.4 Tempo de Inferência

- **YOLO** (ambas variantes): otimizados para velocidade em GPU, processam imagens em dezenas de milissegundos. Adequados para aplicações em tempo real (câmeras de clínicas).
- **CNN do Zero**: também rápida em inferência, mas é apenas classificação — não fornece a localização do objeto (bounding box).

### 8.5 Síntese Final

| Cenário de Uso | Abordagem Recomendada | Justificativa |
|---|---|---|
| **Prova de conceito rápida** | YOLO Tradicional | Zero custo de treino, boa precisão geral |
| **Produto em produção** | YOLO Customizado | Máxima precisão para o domínio específico |
| **Recurso limitado / edge device** | CNN do Zero (otimizada) | Menor footprint de memória |
| **Apenas classificação** | CNN do Zero | Mais simples e interpretável |
| **Detecção + localização** | YOLO (qualquer variante) | CNN não fornece bounding boxes |

### 8.6 Resposta às Perguntas do Projeto

**P: "Uma abordagem é 100% melhor que as outras?"**
> **Não.** Cada abordagem tem seu lugar. O YOLO Customizado vence em precisão para casos especializados, mas requer dataset rotulado e tempo de treino. O YOLO Tradicional é imbatível em agilidade de implantação. A CNN do Zero oferece maior controle sobre a arquitetura e é adequada quando a tarefa é classificação pura e recursos computacionais são limitados.

**P: "Qual indicar ao cliente da FarmTech Solutions?"**
> Para uma clínica veterinária com câmeras de triagem, recomendamos o **YOLO Customizado com 60 épocas** como solução de produção, com a possibilidade de iniciar um MVP com o **YOLO Tradicional** enquanto o dataset customizado é construído e rotulado.

---

*Notebook desenvolvido por **Guilherme Yamada Dantas** (RM 568506) — FIAP 1TIAO — Fase 6 Capítulo 1 — Entrega 2*